# Is the JPEG failure a leftover shortcut? Matching compression history

In GenImage every real image is a JPEG and every fake a PNG. Re-saving everything at JPEG q95 hides the file format, but real images end up compressed **twice** and fakes only **once**. A detector can still learn "more compression traces = real", which would explain why recall on modern fakes falls from 93% to 7% after JPEG-50.

This notebook removes that difference: it estimates the JPEG quality the real photos were originally saved at, gives every fake a first JPEG pass drawn from the same distribution, then applies the same q95 re-save. Features are extracted for these **matched** fakes; real images keep their existing features (they already went through JPEG then q95).

In [ ]:
"""Kaggle GPU job: extract frozen features once, so everything else runs on CPU.

For every sampled image and every *view* (perturbation), store:
  - CLIP ViT-L/14 image embedding (768-d)        [UnivFD-style representation]
  - DINOv2 ViT-B/14 CLS embedding (768-d)
  - 64-d low-level spectrum feature (radial FFT profile of the noise residual)

Views:
  raw      : file as stored (GenImage: real=JPEG, fake=PNG -> format shortcut!)
  jpeg95   : every image re-encoded JPEG q95 (removes the format shortcut)
  jpeg75, jpeg50, resize50, blur : realistic degradations (test split only)

Output: /kaggle/working/feats/{meta.csv, <model>_<view>.npy, spec_<view>.npy}
"""

import io
import os
import random
import re
import sys
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter

N_TRAIN_PER_CLASS = int(os.getenv("N_TRAIN", 1000))
N_TEST_PER_CLASS = int(os.getenv("N_TEST", 500))
OUT = "/kaggle/working/feats"
os.makedirs(OUT, exist_ok=True)
random.seed(0)


def find_subsets(root="/kaggle/input"):
    subs = {}
    for dp, dns, _ in os.walk(root):
        base = os.path.basename(dp)
        if base.lower().startswith("imagenet_") and {"train", "val"} & set(dns):
            subs[base.split("_")[-1].lower()] = dp
    return subs


def list_imgs(d):
    return sorted(os.path.join(d, f) for f in os.listdir(d)) if os.path.isdir(d) else []


IMG_EXT = (".png", ".jpg", ".jpeg", ".webp")


def build_external_meta(n=int(os.getenv("N_EXT", 500))):
    """Fakes from generators never seen in training (Synthbuster + Nano Banana). All label=1."""
    rows = []
    for dp, dns, fns in os.walk("/kaggle/input"):
        imgs = sorted(f for f in fns if f.lower().endswith(IMG_EXT))
        if len(imgs) < 50:
            continue
        low = dp.lower()
        if "synthbuster" in low:
            gen = "sb_" + os.path.basename(dp).lower()
        elif "nano banana" in low or "nano-banana" in low:
            gen = "nanobanana2_unverified"
        else:
            continue
        random.shuffle(imgs)
        for f in imgs[:n]:
            rows.append({"path": os.path.join(dp, f), "generator": gen, "split": "test", "label": 1,
                         "ext": os.path.splitext(f)[1].lower()})
    meta = pd.DataFrame(rows)
    print(meta.groupby("generator").size(), flush=True)
    return meta


def build_meta():
    if os.getenv("EXTERNAL"):
        return build_external_meta()
    subs = find_subsets()
    print("found subsets:", subs, flush=True)
    rows = []
    for gen, d in sorted(subs.items()):
        for split, n in (("train", N_TRAIN_PER_CLASS), ("val", N_TEST_PER_CLASS)):
            for cls, label in (("ai", 1), ("nature", 0)):
                files = list_imgs(os.path.join(d, split, cls))
                random.shuffle(files)
                for f in files[:n]:
                    rows.append({"path": f, "generator": gen, "split": "train" if split == "train" else "test",
                                 "label": label, "ext": os.path.splitext(f)[1].lower()})
    meta = pd.DataFrame(rows)
    print(meta.groupby(["generator", "split", "label"]).size().unstack(), flush=True)
    print("extensions by label:\n", meta.groupby(["label", "ext"]).size(), flush=True)
    return meta


def jpeg(img, q):
    b = io.BytesIO(); img.save(b, "JPEG", quality=q); b.seek(0)
    return Image.open(b).convert("RGB")


VIEWS = {
    "raw": lambda im: im,
    "jpeg95": lambda im: jpeg(im, 95),
    "jpeg75": lambda im: jpeg(im, 75),
    "jpeg50": lambda im: jpeg(im, 50),
    "resize50": lambda im: jpeg(im.resize((max(1, im.width // 2), max(1, im.height // 2)), Image.BICUBIC), 95),
    "blur": lambda im: jpeg(im.filter(ImageFilter.GaussianBlur(1.0)), 95),
}
TEST_ONLY = {"jpeg75", "jpeg50", "resize50", "blur"}


def center_crop(im, s=224):
    if min(im.size) < s:
        r = s / min(im.size)
        im = im.resize((round(im.width * r), round(im.height * r)), Image.BICUBIC)
    l, t = (im.width - s) // 2, (im.height - s) // 2
    return im.crop((l, t, l + s, t + s))


def spectrum_feat(im, bins=64):
    g = np.asarray(im.convert("L"), dtype=np.float32) / 255.0
    # noise residual: image minus 3x3 box blur, emphasises generator upsampling artifacts
    k = np.ones((3, 3), np.float32) / 9
    from numpy.lib.stride_tricks import sliding_window_view
    pad = np.pad(g, 1, mode="reflect")
    res = g - (sliding_window_view(pad, (3, 3)) * k).sum((-1, -2))
    f = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(res))))
    h, w = f.shape
    yy, xx = np.indices(f.shape)
    r = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2)
    r = (r / r.max() * (bins - 1)).astype(int)
    prof = np.bincount(r.ravel(), f.ravel(), bins) / np.maximum(np.bincount(r.ravel(), minlength=bins), 1)
    return prof.astype(np.float32)


MEAN = {"clip": (0.4815, 0.4578, 0.4082), "dino": (0.485, 0.456, 0.406)}
STD = {"clip": (0.2686, 0.2613, 0.2758), "dino": (0.229, 0.224, 0.225)}


def to_tensor(crops, kind, dev):
    x = torch.from_numpy(np.stack([np.asarray(c, np.float32) / 255.0 for c in crops])).permute(0, 3, 1, 2)
    m = torch.tensor(MEAN[kind]).view(1, 3, 1, 1); s = torch.tensor(STD[kind]).view(1, 3, 1, 1)
    return ((x - m) / s).to(dev, torch.float16)




## 1. Estimate the original JPEG quality of the real images
From the luminance quantisation table, using the standard IJG scaling.

In [ ]:
import json
from PIL import JpegImagePlugin
STD_LUMA = np.array([16,11,10,16,24,40,51,61,12,12,14,19,26,58,60,55,14,13,16,24,40,57,69,56,14,17,22,29,51,87,80,62,
 18,22,37,56,68,109,103,77,24,35,55,64,81,104,113,92,49,64,78,87,103,121,120,101,72,92,95,98,112,100,103,99])
def est_quality(path):
    im = Image.open(path)
    q = getattr(im, "quantization", None)
    if not q: return None
    t = np.array(q[0], dtype=float)
    # IJG: table = (std*scale+50)/100 ; scale = 5000/Q (Q<50) or 200-2Q
    scale = np.median(t * 100 / STD_LUMA)
    return int(np.clip(round(5000 / scale) if scale > 100 else round((200 - scale) / 2), 1, 100))

meta = build_meta()
reals = meta[(meta.label == 0)]
qs = [est_quality(p) for p in reals.sample(2000, random_state=0).path]
qs = np.array([q for q in qs if q])
print("real-image JPEG quality: median", np.median(qs), "p10/p90", np.percentile(qs, [10, 90]))
print(pd.Series(qs).value_counts().head(10))

## 2. Extract features for fakes with a matched compression history

In [ ]:
from transformers import AutoModel, CLIPVisionModelWithProjection
from concurrent.futures import ThreadPoolExecutor
rng = np.random.default_rng(0)
fakes = meta[meta.label == 1]
first_q = rng.choice(qs, size=len(fakes))  # sample from the real-image quality distribution
d0 = "cuda:0"; d1 = "cuda:1" if torch.cuda.device_count() > 1 else d0
clip = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14", torch_dtype=torch.float16).to(d0).eval()
dino = AutoModel.from_pretrained("facebook/dinov2-base", torch_dtype=torch.float16).to(d1).eval()

def load_matched(args):
    p, q = args
    im = jpeg(jpeg(Image.open(p).convert("RGB"), int(q)), 95)  # first pass like the reals, then the same q95 re-save
    return center_crop(im)

def views_of(crop):  # same test-time degradations as the main study, applied after matching
    return {"matched": crop}

N = len(meta); idx = fakes.index.values
C = np.zeros((N, 768), np.float16); D = np.zeros((N, 768), np.float16); S = np.zeros((N, 64), np.float32); have = np.zeros(N, bool)
pool = ThreadPoolExecutor(8); t = time.time()
for i in range(0, len(idx), 128):
    rows = idx[i:i+128]
    crops = list(pool.map(load_matched, zip(meta.loc[rows, "path"], first_q[i:i+128])))
    with torch.no_grad():
        C[rows] = clip(pixel_values=to_tensor(crops, "clip", d0)).image_embeds.float().cpu().numpy()
        D[rows] = dino(pixel_values=to_tensor(crops, "dino", d1)).pooler_output.float().cpu().numpy()
    S[rows] = np.stack([spectrum_feat(c) for c in crops]); have[rows] = True
    if i % 2560 == 0: print(i, len(idx), f"{time.time()-t:.0f}s", flush=True)
for k, a in [("clip", C), ("dino", D), ("spec", S), ("have", have)]:
    np.save(f"{OUT}/{k}_matched.npy", a)
meta.to_csv(f"{OUT}/meta.csv", index=False)
json.dump({"real_quality_median": float(np.median(qs)), "real_quality_hist": pd.Series(qs).value_counts().to_dict()},
          open(f"{OUT}/real_quality.json", "w"), default=int)
print("done", have.sum(), f"{time.time()-t:.0f}s")

Analysis runs on CPU in the repo: `python src/matched.py`. It trains detectors with the matched fakes and re-runs the JPEG-50 tests on unseen GenImage generators and on Synthbuster.